# Does a High Probability Mean the Model Is Right?

**Notebook 3 of the main path.** Notebooks 1 and 2 showed that an AI
language model always deals in percentages — real ones, computed the same
way every time. This notebook asks the question that actually matters for
using these tools at work: **can something be fluent and probable, and
still not be true?**

We'll use a real, concrete example an oilfield professional would
recognize immediately: a specific number in a field report — a pressure
test value — and test, with the real model, whether a confident-sounding
number can appear without ever being checked against anything real.

## What you'll be able to answer by the end

1. Can a language model sound completely confident and still be wrong?
2. What does it mean that the model has never seen any specific well's
   real, approved documentation?
3. If a number were really being looked up from a real source, what
   should *not* happen when we change an unrelated detail of the
   question — and what actually happens?
4. Does this only happen with invented facts, or can it happen even with
   a genuinely well-established, true fact?
5. What should this change about how you use an AI tool for anything
   involving a specific number or specification?


## 1. What problem are we investigating?

Imagine you saw this line in a field report:

> "Field notes: at Well A-12, the crew pressure tested the 9-5/8 inch
> casing string, held steady at 1,000 psi."

Nothing about that sentence looks wrong. It's grammatically clean, it uses
real oilfield terminology correctly, and the number is a perfectly
reasonable-looking pressure test value. If an AI tool wrote that sentence
for you, would you have any reason to doubt it?

This notebook shows, with the real model, exactly how that sentence gets
produced — and what that process can and can't tell you about whether the
number in it is actually correct for that well.


## Installation (run once)

If you already set up the environment for notebooks 1-2, you don't need to
do anything further. Otherwise, uncomment and run the cell below once.


In [1]:
# Run this once. After the packages are installed you can leave this commented out.
# %pip install torch transformers accelerate pandas matplotlib numpy
print("If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.")


If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.


## 2. Load the real AI model

Same model, same computer setup as notebooks 1-2. As before, **you don't
need to understand the next code cell; just run it.**


In [2]:
import random
import sys

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PRIMARY_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PRIMARY_MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"  # pinned for reproducibility
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # used only if the primary model fails to load
FALLBACK_MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"  # pinned for reproducibility

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

DTYPE = torch.float16 if DEVICE in ("mps", "cuda") else torch.float32

print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Selected device:      {DEVICE}")
print(f"Selected dtype:       {DTYPE}")


Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Selected device:      mps
Selected dtype:       torch.float16


In [3]:
def load_model(model_name: str = PRIMARY_MODEL_NAME):
    '''Load a causal language model and its tokenizer onto the selected device.

    Falls back to FALLBACK_MODEL_NAME if the primary model cannot be loaded,
    and always reports which model actually ended up running.
    '''
    try:
        tok = AutoTokenizer.from_pretrained(model_name, revision=PRIMARY_MODEL_REVISION)
        mdl = AutoModelForCausalLM.from_pretrained(model_name, revision=PRIMARY_MODEL_REVISION, dtype=DTYPE, use_safetensors=True)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, model_name
    except Exception as exc:  # noqa: BLE001 - report and fall back, don't crash the notebook
        print(f"Could not load '{model_name}' ({exc}). Falling back to '{FALLBACK_MODEL_NAME}'.")
        tok = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME, revision=FALLBACK_MODEL_REVISION)
        mdl = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL_NAME, revision=FALLBACK_MODEL_REVISION, dtype=DTYPE, use_safetensors=True)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, FALLBACK_MODEL_NAME


tokenizer, model, MODEL_NAME = load_model()
print(f"\nModel actually loaded and used in this notebook: {MODEL_NAME}")



Model actually loaded and used in this notebook: Qwen/Qwen2.5-1.5B-Instruct


## 3. Reusing what earlier notebooks already built

A generation function like the one in notebook 2 — repeatedly asking the
model for its single top answer and adding it to the text. We'll use this
"always play it safe" (greedy) rule throughout this notebook because it's
fully predictable: the same input always produces the same output, which
makes it the fairest way to test whether a number changes for a real
reason or not.


In [4]:
def get_next_token_distribution(prompt: str):
    '''Run `prompt` through the model and return (input_ids, logits, probabilities) for the next token.'''
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**enc)
    logits_vec = out.logits[0, -1, :]
    probs_vec = torch.softmax(logits_vec, dim=-1)
    return enc["input_ids"][0], logits_vec, probs_vec


def generate_greedy(prompt: str, n_new_tokens: int) -> str:
    '''Repeatedly take the model's single top answer and add it to the text.'''
    text = prompt
    for _ in range(n_new_tokens):
        _, logits_vec, _ = get_next_token_distribution(text)
        next_id = int(torch.argmax(logits_vec).item())
        text += tokenizer.decode([next_id])
    return text


def show(df: pd.DataFrame):
    '''Display a table without pandas' default row-number column on the left, which isn't real data.'''
    display(df.style.hide(axis="index"))


## 4. A confident-sounding number, generated live

Let's actually generate the sentence from Section 1, live, with the real
model — starting from an unfinished field note and letting the model
write the rest, one real word at a time.


In [5]:
PROMPT = "Field notes: at Well A-12, the crew pressure tested the 9-5/8 inch casing string, held steady at"

result = generate_greedy(PROMPT, n_new_tokens=8)
print(result)
print("(Well A-12 is fictional -- this number is not a real engineering spec.)")


Field notes: at Well A-12, the crew pressure tested the 9-5/8 inch casing string, held steady at 1,000 psi.
(Well A-12 is fictional -- this number is not a real engineering spec.)


**What this shows:** a specific, plausible-sounding pressure value,
written with complete grammatical confidence. Nothing about *how* it reads
tells you whether that number came from anywhere real. There is no Well
A-12 anywhere in the huge amount of text this model learned from — we made
that name up for this notebook — so whatever number appears cannot be a
fact this model actually knows about that well. The question is: where did
it come from, and can we trust it?


## 5. The real test: if this were a real fact, would it change?

If the model were somehow drawing on a real, specific fact about a real
well, then changing an arbitrary, fictional label — like *which* well
we're talking about — shouldn't change a value retrieved from the same
approved source.

Let's check, with six different, equally fictional well names, using the
exact same sentence otherwise.


In [6]:
wells = [
    "Well A-12",
    "Well B-7",
    "the North Pad well",
    "Well 204",
    "the offshore platform well",
    "Well C-3",
]

well_rows = []
for well in wells:
    prompt = f"Field notes: at {well}, the crew pressure tested the 9-5/8 inch casing string, held steady at"
    completion = generate_greedy(prompt, n_new_tokens=8)
    generated_part = completion[len(prompt):]
    well_rows.append({"Well": well, "Model's generated ending": generated_part})

well_table = pd.DataFrame(well_rows)
show(well_table)


Well,Model's generated ending
Well A-12,"1,000 psi."
Well B-7,"1,000 psi."
the North Pad well,"1,000 psi."
Well 204,"1,000 psi."
the offshore platform well,"10,000 psi"
Well C-3,"1,000 psi."


**What this shows — read the real table above.** While building this
notebook, five of the six completely fictional wells produced the exact
same value ("1,000 psi"), and one — "the offshore platform well" —
produced a different one ("10,000 psi"), ten times larger. Neither result
should be read as "the model knows the real answer for 5 wells but not the
sixth." **None of these wells exist**, so there is no real answer for the
model to know in the first place. What this table actually demonstrates is
two things at once:

1. The *consistency* across five different, made-up well names shows the
   model isn't recalling a fact specific to any one of them — it's
   defaulting to whatever value is most common in similar-sounding
   sentences it saw during training.
2. The *one exception* shows that "most common default" isn't even a
   stable rule — an unrelated detail (the word "offshore") was enough to
   change the generated number by 10x. A real, retrieved fact wouldn't
   behave like that.


## 6. Does this only happen with made-up facts?

You might reasonably ask: sure, there's no real "Well A-12," so of course
the model can't know a real answer. What about something that genuinely
has one correct, well-established value — the kind of fact that appears
constantly, worded many different ways, throughout the vast amount of text
this model learned from?

One barrel of oil is a fixed, internationally standard unit: **42 US
gallons**, which is about **159 liters**. Let's ask the same real question
three different, equally reasonable ways.


In [7]:
fact_prompts = [
    "One barrel of oil is equal to",
    "In the oilfield, one barrel is defined as",
    "By definition, one barrel of crude oil equals",
]

for prompt in fact_prompts:
    completion = generate_greedy(prompt, n_new_tokens=8)
    print(completion)


One barrel of oil is equal to 42 liters. If a truck


In the oilfield, one barrel is defined as 159 liters. If a


By definition, one barrel of crude oil equals 42 US gallons. If a


**What this shows — again, read the real results above rather than
assuming.** While building this notebook, two of these three phrasings
gave a correct answer (either "159 liters" or "42 US gallons" — both
correct, just in different units). One gave **"42 liters"** — which mixes
the *number* from the correct gallons answer with the *unit* from the
correct liters answer, producing something that reads just as fluently and
confidently as the other two, but is wrong.

This is the honest, complete picture: even for a real, unambiguous,
universally-agreed-upon fact, changing only the *phrasing* of the question
was enough to produce one wrong answer alongside two right ones — and
nothing about how any of the three sentences *read* would tell you which
one to trust.


## 7. What this means: probable is not the same thing as correct

Put the two experiments together, and the lesson is exactly the one this
series has been building toward since notebook 1:

- **Probability measures how common a pattern of words is**, based on
  everything the model was trained on. It does not measure whether a
  specific claim has been checked against a specific, real source.
- **A model has no way to "know" it lacks the real answer.** There is no
  internal flag that lights up when a well doesn't exist, or when a
  question has no single correct answer anywhere in the text it learned
  from. It produces its most probable-sounding continuation regardless.
- **Fluency and confidence are not evidence of correctness.** All of the
  sentences in this notebook — the ones with a fabricated number, and the
  one wrong unit for a real fact — read exactly as smoothly and
  confidently as the correct ones.
- **This gets more dangerous, not less, as questions get more specific.**
  A generic, widely-repeated fact (like the barrel conversion) still has a
  real chance of coming out right. A question that can only be answered by
  one specific, non-public document — like an actual approved well
  programme — has no such chance, no matter how the sentence reads.


## 8. So what should you actually do?

The honest conclusion from Sections 4-7 is not "never trust an AI tool for
numbers." It's more specific than that:

- For a **specific, safety- or operations-critical number** (a test
  pressure, a torque spec, a depth, a rating) — never take a model's
  output as the source. Verify it against the actual approved document for
  that specific well or job, every time. This notebook just showed you,
  with real evidence, exactly why that number can't be trusted on its own.
- For a **general, well-established fact**, an AI tool's answer has a real
  chance of being right — but as Section 6 showed, not a guarantee, and
  you may not be able to tell which case you're in just by reading the
  answer.
- If you want an AI tool that only answers from real, specific,
  verifiable engineering documents — rather than from generalized patterns
  learned from unrelated text — that requires deliberately connecting it
  to those documents and having it point to exactly where an answer came
  from. That approach (often called **retrieval-augmented generation**,
  or "grounding") is exactly what notebook 4 does next.


## 9. Key lessons

1. A language model can produce a specific, fluent, confident-sounding
   number with no real source behind it at all.
2. Changing an unrelated detail of a question is a real, concrete way to
   test whether an answer is a retrieved fact or a probable-sounding
   pattern — we did exactly that, with real results, in Section 5.
3. This isn't limited to invented scenarios: even a genuinely correct,
   universally-agreed-upon fact can come out wrong depending on how the
   question is phrased (Section 6).
4. Fluency and confidence are not signals of correctness — every sentence
   in this notebook, right or wrong, read equally smoothly.
5. The more specific and non-public the real answer would have to be (a
   particular well's approved programme, as opposed to a general industry
   fact), the less reason there is to trust a model's unverified answer.
6. **Probable is not the same thing as correct. Fluent is not the same
   thing as verified.**


## 10. Try your own example

Pick a specific-number question from your own work, phrase it two or three
different ways, and see whether the model's answer stays the same. Change
only the line below.


In [8]:
user_prompt = "Torque the connection to a final make-up torque of"  # <-- change this line

user_result = generate_greedy(user_prompt, n_new_tokens=10)
print(user_result)


Torque the connection to a final make-up torque of 100 Nm. The connection is


## 11. Optional exercises

1. In Section 5, add two more fictional well names of your own. Does the
   pattern (mostly one value, with occasional exceptions) hold up?
2. In Section 6, try a third phrasing of the barrel-conversion question of
   your own design. Does it come out right or wrong?
3. Pick a different, genuinely well-established oilfield fact (for
   example, a standard unit conversion or an API definition) and run the
   same three-phrasings test as Section 6.
4. In Section 10, ask the same question with a fictional company name
   changed, or a date added. Does the answer shift?
5. Try increasing `n_new_tokens` in Section 4 to generate a longer field
   note. Does the model add more specific, equally unverifiable details as
   it goes?


## 12. Technical appendix (optional — skip if you like)

**Model and environment actually used in this run** (printed live, not
hard-coded):


In [9]:
print(f"Model:                {MODEL_NAME}")
print(f"Device:               {DEVICE}")
print(f"Dtype:                {DTYPE}")
print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Random seed:          {SEED}")


Model:                Qwen/Qwen2.5-1.5B-Instruct
Device:               mps
Dtype:                torch.float16
Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Random seed:          42


**Why greedy decoding, specifically.** This notebook uses "always
play it safe" (greedy) generation throughout, rather than the dice-rolling
(sampling) from notebook 2. Greedy decoding is fully deterministic — the
same prompt always produces the same output — which makes it the fairest
way to test whether a change in wording causes a real change in the
model's answer, rather than just landing on a different roll of the dice.

**A note on model size.** This notebook uses the same small (1.5B
parameter) model as the rest of the series, for the same reasons: it runs
locally, offline, on a normal laptop. Larger models may behave somewhat
differently on these exact prompts, but the underlying point — that
fluency and probability are not evidence of verification against a real
source — applies to language models generally, not to this model
specifically.
